In [1]:
from dataclasses import dataclass
from openai import APIError, OpenAI
import os
@dataclass
class Provider:
    name: str
    env_var: str
    base_url: str | None
    is_free: bool
    model:str
    
    
Providers = [
    Provider("OPENAI", "OPENAI_API_KEY", None, False, "gpt-4o-mini"),
    Provider("GROQ", "GROQ_API_KEY", "https://api.groq.com/openai/v1", True, "openai/gpt-oss-120b"),
]

def select_provider() -> Provider:
    for provider in Providers:
        if os.getenv(provider.env_var):
            return provider
    raise ValueError("No valid provider found. Please set the appropriate environment variable.")

def build_client(provider: Provider) -> OpenAI:
    api_key = os.getenv(provider.env_var)
    if not api_key:
        raise ValueError(f"{provider.env_var} is not set.")

    if provider.base_url:
        return OpenAI(api_key=api_key, base_url=provider.base_url)

    return OpenAI(api_key=api_key)
    
def have_any_key():
    for provider in Providers:
        if os.getenv(provider.env_var):
            return "Provider: " + provider.name + " is available."
    return "No valid provider found. Please set the appropriate environment variable."

print(have_any_key())

Provider: GROQ is available.


In [2]:
def llm_response(prompt: str):
    provider = select_provider()
    client = build_client(provider)
    
    response = client.chat.completions.create(
        model=provider.model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200
    )
    
    return response.choices[0].message.content.strip()

In [4]:
prompt = "Who is the president of the United States?"
try:
    response = llm_response(prompt)
    print(f"Response: {response}")
except (ValueError, APIError) as e:
    print(f"Error: {e}")

Response: 


In [12]:
def chat_reply(messages: List[Dict]):
    provider = select_provider()
    client = build_client(provider)
    
    response = client.chat.completions.create(
        model=provider.model,
        messages=messages,
        max_tokens=1000
    )
    
    return response.choices[0].message.content.strip()

In [14]:
conversation:List[Dict] = []

In [15]:
conversation.append({"role": "user", "content": "Who is the prime minister of India?"}) 

In [17]:
reply_from_llm = chat_reply(conversation)
conversation.append({"role": "assistant", "content": reply_from_llm})

In [19]:
conversation

[{'role': 'user', 'content': 'Who is the prime minister of India?'},
 {'role': 'assistant',
  'content': 'As of the most recent information available,\u202f**Narendra Modi**\u202fhas been serving as the Prime Minister of India. He has held the office since May\u202f2014.'},
 {'role': 'assistant',
  'content': 'Narendra\u202fModi has been the Prime Minister of India since\u202fMay\u202f2014, and he continues to hold the office.'}]

In [21]:
conversation.append({"role": "user", "content": "What is gdp of India?"})

In [23]:
reply_from_llm = chat_reply(conversation)
conversation.append({"role": "assistant", "content": reply_from_llm})


In [24]:
conversation

[{'role': 'user', 'content': 'Who is the prime minister of India?'},
 {'role': 'assistant',
  'content': 'As of the most recent information available,\u202f**Narendra Modi**\u202fhas been serving as the Prime Minister of India. He has held the office since May\u202f2014.'},
 {'role': 'assistant',
  'content': 'Narendra\u202fModi has been the Prime Minister of India since\u202fMay\u202f2014, and he continues to hold the office.'},
 {'role': 'user', 'content': 'What is gdp of India?'},
 {'role': 'user', 'content': 'What is gdp of India?'},
 {'role': 'assistant',
  'content': '**India’s Gross Domestic Product (GDP) – the latest estimates (2024‑2025)**  \n\n| Metric | Value | Year | Source |\n|--------|-------|------|--------|\n| **Nominal GDP (current‑price USD)** | **≈\u202fUS\u202f$3.73\u202ftrillion** | 2024 (actual) | International Monetary Fund – World Economic Outlook, April\u202f2024 |\n| **Nominal GDP (current‑price USD)** | **≈\u202fUS\u202f$4.04\u202ftrillion** | 2025 (IMF proje